<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/multi_experts_dir.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score
import time

# ── HELPERS classification ────────────────────────────────────────────────
def cls_acc(f, y):
    sg = np.sign(f)
    return float(np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float))))
def cls_auc(f, y):
    try: return float(roc_auc_score((y > 0).astype(int), f))
    except Exception: return float('nan')
def cls_str(f, y):
    return f"acc={cls_acc(f,y):.4f} AUC={cls_auc(f,y):.4f}"

# ══════════════════════════════════════════════════════════════════════════════
# HYPERPARAMÈTRES — une section par type d'expert + une pour la Phase 2.
# n_dirs / batch_dirs / k_loss sont PARTAGÉS (mêmes directions Monte-Carlo pour
# tous les experts, condition nécessaire pour que les termes croisés de la
# Phase 2 aient un sens).
# ══════════════════════════════════════════════════════════════════════════════
params_shared = {
    "n_ambiant": 5, "deg_P": 3, "n_terms_poly": 20,
    "seeds": {1: 59953, 2: 25954, 3: 5605,
              42: 590412, 43: 6852036, 44: 4152403, 11: 24186, 12: 749141},
    "n_train": 200, "n_test": 5000,
    "n_dirs": 50, "batch_dirs": 10,
    "k_loss": 5.0,
}
params_shared["n_unlabeled"] = np.maximum(4000 - params_shared["n_train"], 500)
params_shared["train_center_ratio"] = 0.5 + params_shared["n_train"] / (2 * 4000)

params_gauss = {
    "weights": {0: 0, 1: 1, 2: 0.1, 3: 0.01},
    "lambda_reg": 1e-3, "thresh_factor": 1,
    "sigma_min": 0.1, "sigma_max": 3.0,
    "n_dict": 2000, "n_centres": 800,
    "n_G": 2000,
    "n_experts": 3,
}

params_wnd = {
    "weights": {0: 0, 1: 1, 2: 0.1, 3: 0.01},
    "lambda_reg": 1e-2, "thresh_factor": 10,
    "sigma_min": 0.3, "sigma_max": 3.0,
    "n_dict": 2000, "n_centres": 1000,
    "n_G": 2000,
    "n_experts": 3,
}

params_phase2 = {
    "weights": {0: 0, 1: 1, 2: 0.1, 3: 0.01},
    "lambda_reg": 1e-1, "thresh_factor": 1,
    "n_G_H": 2000,
}

d = params_shared["n_ambiant"]

# ══════════════════════════════════════════════════════════════════════════════
# POLYNÔMES ET CLOUD SUR {Q=0}  (repris tel quel, code déjà validé)
# ══════════════════════════════════════════════════════════════════════════════
def random_sparse_polynomial(dd, degree, n_terms, seed=None):
    rng = np.random.default_rng(seed)
    all_indices = [exp for exp in product(range(degree+1), repeat=dd) if sum(exp) <= degree]
    indices = rng.choice(all_indices, size=n_terms, replace=False)
    coeffs = rng.normal(size=n_terms)
    def P(x):
        y = np.zeros(x.shape[0])
        for c, alpha in zip(coeffs, indices):
            term = np.ones(x.shape[0])
            for j, e in enumerate(alpha):
                if e > 0: term *= x[:, j]**e
            y += c * term
        return y
    zero_val = sum(c for c, alpha in zip(coeffs, indices) if all(e == 0 for e in alpha))
    def P_zero(x): return P(x) - zero_val
    return P_zero, indices, coeffs

def normalize_polynomial(P, indices, coeffs):
    max_c = np.max(np.abs(coeffs))
    if max_c > 0:
        nc = coeffs / max_c
        def Pn(x):
            y = np.zeros(x.shape[0])
            for c, alpha in zip(nc, indices):
                term = np.ones(x.shape[0])
                for j, e in enumerate(alpha):
                    if e > 0: term *= x[:, j]**e
                y += c * term
            return y
        return Pn
    return P

P1, i1, c1 = random_sparse_polynomial(d, params_shared["deg_P"], params_shared["n_terms_poly"], params_shared["seeds"][1])
P2, i2, c2 = random_sparse_polynomial(d, params_shared["deg_P"], params_shared["n_terms_poly"], params_shared["seeds"][2])
P3, i3, c3 = random_sparse_polynomial(d, params_shared["deg_P"], params_shared["n_terms_poly"], params_shared["seeds"][3])
P1 = normalize_polynomial(P1, i1, c1); P2 = normalize_polynomial(P2, i2, c2); P3 = normalize_polynomial(P3, i3, c3)
def Q(x): return P1(x)*P2(x)*P3(x)

def grad_Q_analytical(X):
    eps = 1e-5; dd = X.shape[1]; p1 = P1(X); p2 = P2(X); p3 = P3(X)
    grad = np.zeros_like(X)
    for k in range(dd):
        Xp = X.copy(); Xp[:, k] += eps; Xm = X.copy(); Xm[:, k] -= eps
        dp1 = (P1(Xp)-P1(Xm))/(2*eps); dp2 = (P2(Xp)-P2(Xm))/(2*eps); dp3 = (P3(Xp)-P3(Xm))/(2*eps)
        grad[:, k] = dp1*p2*p3 + p1*dp2*p3 + p1*p2*dp3
    return grad

def project_to_Q_zero(X_init, n_steps=60, tol=1e-4, damp=1.0):
    X = X_init.copy()
    for _ in range(n_steps):
        q = Q(X); gq = grad_Q_analytical(X)
        g2 = np.sum(gq*gq, axis=1, keepdims=True) + 1e-12
        X = X - damp*(q[:, None]*gq)/g2
        X = np.clip(X, 0., 1.)
        if np.abs(Q(X)).max() < tol*0.1:
            break
    return X, np.abs(Q(X)) < tol

def sample_on_Q_zero(n_target, dd, seed=None, max_batches=200):
    rng = np.random.default_rng(seed)
    collected = []; n_col = 0; n_seen = 0; n_ok = 0
    for _ in range(max_batches):
        if n_col >= n_target:
            break
        n_batch = min(max((n_target - n_col)*4, 200), 20000)
        Xi = rng.uniform(0, 1, (int(n_batch), dd))
        Xp, conv = project_to_Q_zero(Xi)
        good = Xp[conv]
        good = good[np.all(np.isfinite(good), axis=1)]
        n_seen += len(Xi); n_ok += int(conv.sum())
        if len(good) > 0:
            collected.append(good); n_col += len(good)
        print(f"  collectés:{n_col}/{n_target} (cumulé {n_ok/max(n_seen,1):.1%})", end='\r')
    print()
    if n_col < n_target:
        raise RuntimeError(f"sample_on_Q_zero: seulement {n_col}/{n_target} points.")
    return np.vstack(collected)[:n_target]

print("Génération du cloud sur {Q=0}...")
X_train = sample_on_Q_zero(params_shared["n_train"], d, seed=params_shared["seeds"][42])
X_test = sample_on_Q_zero(params_shared["n_test"], d, seed=params_shared["seeds"][43])
X_unlabeled = sample_on_Q_zero(params_shared["n_unlabeled"], d, seed=params_shared["seeds"][44])
print(f"  X_train:{X_train.shape}  X_unlabeled:{X_unlabeled.shape}")

P_target1, indicest1, coeffst1 = random_sparse_polynomial(d, 4, params_shared["n_terms_poly"], seed=params_shared["seeds"][11])
P_target2, _, _ = random_sparse_polynomial(d, 4, params_shared["n_terms_poly"], seed=params_shared["seeds"][12])
Ptarget1 = normalize_polynomial(P_target1, indicest1, coeffst1)
Ptarget2 = normalize_polynomial(P_target2, indicest1, coeffst1)

def target_function(X):
    return np.minimum(np.abs(P_target1(X)), 1) + np.minimum(np.abs(P_target2(X)), 1)

y_cont_train = target_function(X_train); y_cont_test = target_function(X_test)
X_all = np.vstack([X_train, X_unlabeled]); N_all = len(X_all)

_med = float(np.median(target_function(X_unlabeled)))
y_train = np.where(y_cont_train >= _med, 1.0, -1.0)
y_test = np.where(y_cont_test >= _med, 1.0, -1.0)
print(f"  cible binarisée au seuil médian={_med:.4f} : "
      f"train (+1)={int((y_train>0).sum())}/{len(y_train)}  "
      f"test (+1)={int((y_test>0).sum())}/{len(y_test)}")
_bal_te = (y_test > 0).mean()
if _bal_te < 0.3 or _bal_te > 0.7:
    print(f"  [!] classes déséquilibrées ({_bal_te:.1%} de +1) — préférer l'AUC.")


def sample_candidates_aniso(X_train, X_unlabeled, n_candidates, train_ratio, rng, dd,
                            sigma_min, sigma_max):
    n_tr = min(int(round(n_candidates*train_ratio)), X_train.shape[0])
    n_ul = min(n_candidates-n_tr, X_unlabeled.shape[0]); parts = []
    if n_tr > 0: parts.append(X_train[rng.choice(X_train.shape[0], n_tr, replace=False)])
    if n_ul > 0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0], n_ul, replace=False)])
    centers = np.vstack(parts)
    log_s = rng.uniform(np.log(sigma_min), np.log(sigma_max), size=(len(centers), dd))
    sigmas = np.exp(log_s)
    return centers, sigmas


# ══════════════════════════════════════════════════════════════════════════════
# GAUSSIENNES ANISOTROPES DIRECTIONNELLES
# phi(x)=exp(-Σ(x_i-c_i)²/(2σ_i²)). Formules directionnelles validées (voir
# historique) — calibration d, d(d+2), d(d+2)(d+4), 9 identique au cas isotrope.
# ══════════════════════════════════════════════════════════════════════════════
def gaussian_features_aniso(X, centers, sigmas):
    diff = X[:, None, :] - centers[None, :, :]
    sq = np.sum(diff**2/sigmas[None, :, :]**2, axis=2)
    return np.exp(-sq/2)

def build_G_gauss_directional_streaming(X_cloud, centers, sigmas, weights, n_dirs,
                                        batch_dirs=10, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    n, d_ = X_cloud.shape
    m = centers.shape[0]
    diff = X_cloud[:, None, :] - centers[None, :, :]
    inv2 = 1.0/sigmas**2
    sq = np.sum(diff**2*inv2[None, :, :], axis=2)
    phi = np.exp(-sq/2)

    w0 = weights.get(0, 0.); w1 = weights.get(1, 0.)
    w2 = weights.get(2, 0.); w3 = weights.get(3, 0.)
    need2 = w2 != 0; need3 = w3 != 0

    G = np.zeros((m, m))
    if w0:
        G += w0*(phi.T@phi)/n
    if need2:
        TR = (-np.sum(inv2, axis=1)[None, :] + np.sum(diff**2*inv2[None, :, :]**2, axis=2))*phi
        TRG = (TR.T@TR)/n
    if need3:
        Csum = -np.sum(inv2, axis=1)
        Dq = np.sum(diff**2*inv2[None, :, :]**2, axis=2)
        coefV = 2*inv2[None, :, :]**2 - (Csum[None, :, None]+Dq[:, :, None])*inv2[None, :, :]
        V = phi[:, :, None]*diff*coefV
        VVG = np.einsum('nid,njd->ij', V, V)/n

    MC1_sum = np.zeros((m, m)); MC2_sum = np.zeros((m, m)); MC3_sum = np.zeros((m, m))
    done = 0
    while done < n_dirs:
        K = min(batch_dirs, n_dirs - done)
        U = rng.normal(size=(n, K, d_))
        U /= np.linalg.norm(U, axis=2, keepdims=True)
        A = np.einsum('nmd,nkd,md->nmk', diff, U, inv2)
        B = np.einsum('nkd,md->nmk', U**2, inv2)
        gp = -A; gpp = -B
        if w1:
            D1 = gp*phi[:, :, None]
            D1f = D1.transpose(0, 2, 1).reshape(-1, m)
            MC1_sum += D1f.T@D1f
            del D1, D1f
        if need2 or need3:
            D2 = (gpp+gp**2)*phi[:, :, None]
            if need2:
                D2f = D2.transpose(0, 2, 1).reshape(-1, m)
                MC2_sum += D2f.T@D2f
                del D2f
            if need3:
                D3 = (3*gp*gpp+gp**3)*phi[:, :, None]
                D3f = D3.transpose(0, 2, 1).reshape(-1, m)
                MC3_sum += D3f.T@D3f
                del D3, D3f
            del D2
        del U, A, B, gp, gpp
        done += K

    if w1: G += w1*d_*MC1_sum/(n*n_dirs)
    if need2:
        MC2 = MC2_sum/(n*n_dirs)
        G += w2*(d_*(d_+2)*MC2 - TRG)/2
    if need3:
        MC3 = MC3_sum/(n*n_dirs)
        G += w3*(d_*(d_+2)*(d_+4)*MC3 - 9*VVG)/6
    return G


# ══════════════════════════════════════════════════════════════════════════════
# WENDLAND ANISOTROPES DIRECTIONNELS
# φ(r), r=‖(x-c)/σ‖. Formules directionnelles via règle de la chaîne (φ',φ'',φ'''
# BRUTES, dérivées symboliques). Validées (voir historique).
# Profil choisi selon l'ordre max requis PAR L'UN OU L'AUTRE des poids
# Wendland-Phase1 ET Phase2 (Phase 2 doit pouvoir dériver les experts Wendland
# à son propre ordre, potentiellement différent de celui utilisé pour les fitter).
# ══════════════════════════════════════════════════════════════════════════════
def _wnd_raw_profile(max_order):
    k = max(1, int(np.ceil(max_order/2)))
    if k == 1:      # C²
        return (lambda r,rp: rp**4*(4*r+1),
                lambda r,rp: -20*r*rp**3,
                lambda r,rp: 20*rp**2*(4*r-1),
                lambda r,rp: 120*rp*(1-2*r))
    elif k == 2:    # C⁴
        return (lambda r,rp: rp**6*(35*r**2+18*r+3)/3,
                lambda r,rp: -56*r*rp**5*(5*r+1)/3,
                lambda r,rp: 56*rp**4*(35*r**2-4*r-1)/3,
                lambda r,rp: -560*r*rp**3*(7*r-3))
    else:           # C⁶
        return (lambda r,rp: rp**8*(32*r**3+25*r**2+8*r+1),
                lambda r,rp: -22*r*rp**7*(16*r**2+7*r+1),
                lambda r,rp: 22*rp**6*(160*r**3+15*r**2-6*r-1),
                lambda r,rp: -1584*r*rp**5*(20*r**2-5*r-1))

_wnd_orders_all = ([o for o, w in params_wnd["weights"].items() if w != 0] +
                   [o for o, w in params_phase2["weights"].items() if w != 0])
_wnd_max_order = max(_wnd_orders_all) if _wnd_orders_all else 1
WND_PHI, WND_P1, WND_P2, WND_P3 = _wnd_raw_profile(_wnd_max_order)
print(f"Profil Wendland C{2*max(1,int(np.ceil(_wnd_max_order/2)))} "
      f"(ordre max requis, Phase 1 wnd ∪ Phase 2 = {_wnd_max_order})")

def wnd_features_aniso(X, centers, sigmas):
    diff = X[:, None, :] - centers[None, :, :]
    inv2 = 1.0/sigmas**2
    r = np.sqrt(np.sum(diff**2*inv2[None, :, :], axis=2))
    rp = np.maximum(1.-r, 0.)
    return WND_PHI(r, rp)

def build_G_wnd_directional_streaming(X_cloud, centers, sigmas, weights, n_dirs,
                                      batch_dirs=10, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    n, d_ = X_cloud.shape
    m = centers.shape[0]
    diff = X_cloud[:, None, :] - centers[None, :, :]
    inv2 = 1.0/sigmas**2
    r = np.sqrt(np.sum(diff**2*inv2[None, :, :], axis=2))
    rp = np.maximum(1.-r, 0.)
    r_safe = np.maximum(r, 1e-9)
    p0, p1v, p2v, p3v = WND_PHI(r, rp), WND_P1(r, rp), WND_P2(r, rp), WND_P3(r, rp)

    w0 = weights.get(0, 0.); w1 = weights.get(1, 0.)
    w2 = weights.get(2, 0.); w3 = weights.get(3, 0.)
    need2 = w2 != 0; need3 = w3 != 0

    G = np.zeros((m, m))
    if w0:
        G += w0*(p0.T@p0)/n

    if need2 or need3:
        Q4 = np.sum(diff**2*inv2[None, :, :]**2, axis=2)
        S0 = np.sum(inv2, axis=1)
    if need2:
        TR = p2v*Q4/r_safe**2 + p1v*(S0[None, :]/r_safe - Q4/r_safe**3)
        TR = np.where(r > 1e-9, TR, 0.)
        TRG = (TR.T@TR)/n
    if need3:
        dr = diff*inv2[None, :, :]/r_safe[:, :, None]
        dQ4 = 2*diff*inv2[None, :, :]**2
        V = (p3v[:, :, None]*dr*Q4[:, :, None]/r_safe[:, :, None]**2
            + p2v[:, :, None]*(dQ4/r_safe[:, :, None]**2 - 2*Q4[:, :, None]*dr/r_safe[:, :, None]**3)
            + p2v[:, :, None]*dr*(S0[None, :, None]/r_safe[:, :, None] - Q4[:, :, None]/r_safe[:, :, None]**3)
            + p1v[:, :, None]*(-S0[None, :, None]*dr/r_safe[:, :, None]**2
                               - dQ4/r_safe[:, :, None]**3
                               + 3*Q4[:, :, None]*dr/r_safe[:, :, None]**4))
        V = np.where(r[:, :, None] > 1e-9, V, 0.)
        VVG = np.einsum('nid,njd->ij', V, V)/n

    MC1_sum = np.zeros((m, m)); MC2_sum = np.zeros((m, m)); MC3_sum = np.zeros((m, m))
    done = 0
    while done < n_dirs:
        K = min(batch_dirs, n_dirs - done)
        U = rng.normal(size=(n, K, d_))
        U /= np.linalg.norm(U, axis=2, keepdims=True)
        A = np.einsum('nmd,nkd,md->nmk', diff, U, inv2)
        B = np.einsum('nkd,md->nmk', U**2, inv2)
        r0 = r_safe[:, :, None]
        rprime = A/r0
        rpprime = -A**2/r0**3 + B/r0
        f1 = p1v[:, :, None]*rprime
        if w1:
            D1f = f1.transpose(0, 2, 1).reshape(-1, m)
            MC1_sum += D1f.T@D1f
            del D1f
        if need2 or need3:
            f2 = p2v[:, :, None]*rprime**2 + p1v[:, :, None]*rpprime
            if need2:
                D2f = f2.transpose(0, 2, 1).reshape(-1, m)
                MC2_sum += D2f.T@D2f
                del D2f
            if need3:
                rppprime = 3*A*(A**2 - B*r0**2)/r0**5
                f3 = p3v[:, :, None]*rprime**3 + 3*p2v[:, :, None]*rprime*rpprime + p1v[:, :, None]*rppprime
                D3f = f3.transpose(0, 2, 1).reshape(-1, m)
                MC3_sum += D3f.T@D3f
                del D3f, f3, rppprime
            del f2
        del U, A, B, rprime, rpprime, f1
        done += K

    if w1: G += w1*d_*MC1_sum/(n*n_dirs)
    if need2:
        MC2 = MC2_sum/(n*n_dirs)
        G += w2*(d_*(d_+2)*MC2 - TRG)/2
    if need3:
        MC3 = MC3_sum/(n*n_dirs)
        G += w3*(d_*(d_+2)*(d_+4)*MC3 - 9*VVG)/6
    return G


# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 : experts gaussiens anisotropes INDÉPENDANTS
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nPhase 1 (gaussien) : {params_gauss['n_experts']} experts indépendants...")

experts = []   # liste unifiée : {'type','centers','sigmas','coeffs'}
F_train_list = []; F_test_list = []
times_p1 = []

for e in range(params_gauss["n_experts"]):
    t0 = time.time()
    rng = np.random.default_rng(seed=100+e)
    candidates, sigmas_cand = sample_candidates_aniso(
        X_train, X_unlabeled, params_gauss["n_dict"], params_shared["train_center_ratio"],
        rng, d, params_gauss["sigma_min"], params_gauss["sigma_max"])

    A_cand = gaussian_features_aniso(X_train, candidates, sigmas_cand)
    corr = (A_cand.T@y_train)/len(X_train)
    scores = corr**2

    k = min(params_gauss["n_centres"], len(candidates))
    top_k = np.argsort(scores)[-k:]
    centers = candidates[top_k]; sigmas_sel = sigmas_cand[top_k]
    A = A_cand[:, top_k]
    Atest = gaussian_features_aniso(X_test, centers, sigmas_sel)

    n_G = min(N_all, params_gauss["n_G"])
    idx_G = rng.choice(N_all, size=n_G, replace=False)
    G = build_G_gauss_directional_streaming(
        X_all[idx_G], centers, sigmas_sel, params_gauss["weights"],
        n_dirs=params_shared["n_dirs"], batch_dirs=params_shared["batch_dirs"],
        rng=np.random.default_rng(9000+e))

    n = A.shape[0]
    M = (A.T@A)/n + params_gauss["lambda_reg"]*G
    rhs = (A.T@y_train)/n
    eigvals, eigvecs = np.linalg.eigh(M)
    thresh = max(params_gauss["lambda_reg"]*params_gauss["thresh_factor"], 1e-15)
    mask = eigvals > thresh
    V = eigvecs[:, mask]; S = eigvals[mask]; coeffs = V@((V.T@rhs)/S)

    f_tr = A@coeffs; f_te = Atest@coeffs
    loss_data = np.mean((y_train - f_tr)**2)
    loss_reg = params_gauss["lambda_reg"]*coeffs@G@coeffs
    t1 = time.time(); times_p1.append(t1-t0)

    F_train_list.append(f_tr); F_test_list.append(f_te)
    experts.append({'type': 'gauss', 'centers': centers, 'sigmas': sigmas_sel, 'coeffs': coeffs})

    print(f"  gauss {e+1}/{params_gauss['n_experts']} | rang={mask.sum():3d}/{k} | "
          f"loss={loss_data+loss_reg:.6f} (data={loss_data:.6f} reg={loss_reg:.6f}) | "
          f"{cls_str(f_te, y_test)} | {times_p1[-1]:.1f}s")


# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 : experts Wendland anisotropes INDÉPENDANTS
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nPhase 1 (Wendland) : {params_wnd['n_experts']} experts indépendants...")

for e in range(params_wnd["n_experts"]):
    t0 = time.time()
    rng = np.random.default_rng(seed=500+e)
    candidates, sigmas_cand = sample_candidates_aniso(
        X_train, X_unlabeled, params_wnd["n_dict"], params_shared["train_center_ratio"],
        rng, d, params_wnd["sigma_min"], params_wnd["sigma_max"])

    A_cand = wnd_features_aniso(X_train, candidates, sigmas_cand)
    corr = (A_cand.T@y_train)/len(X_train)
    scores = corr**2

    k = min(params_wnd["n_centres"], len(candidates))
    top_k = np.argsort(scores)[-k:]
    centers = candidates[top_k]; sigmas_sel = sigmas_cand[top_k]
    A = A_cand[:, top_k]
    Atest = wnd_features_aniso(X_test, centers, sigmas_sel)

    n_G = min(N_all, params_wnd["n_G"])
    idx_G = rng.choice(N_all, size=n_G, replace=False)
    G = build_G_wnd_directional_streaming(
        X_all[idx_G], centers, sigmas_sel, params_wnd["weights"],
        n_dirs=params_shared["n_dirs"], batch_dirs=params_shared["batch_dirs"],
        rng=np.random.default_rng(9500+e))

    n = A.shape[0]
    M = (A.T@A)/n + params_wnd["lambda_reg"]*G
    rhs = (A.T@y_train)/n
    eigvals, eigvecs = np.linalg.eigh(M)
    thresh = max(params_wnd["lambda_reg"]*params_wnd["thresh_factor"], 1e-7)
    mask = eigvals > thresh
    V = eigvecs[:, mask]; S = eigvals[mask]; coeffs = V@((V.T@rhs)/S)

    f_tr = A@coeffs; f_te = Atest@coeffs
    loss_data = np.mean((y_train - f_tr)**2)
    loss_reg = params_wnd["lambda_reg"]*coeffs@G@coeffs
    t1 = time.time(); times_p1.append(t1-t0)

    if np.max(np.abs(f_tr)) < 1e-12:
        print(f"  [!] wnd {e+1} quasi nul sur le train (support compact non couvert)")

    F_train_list.append(f_tr); F_test_list.append(f_te)
    experts.append({'type': 'wnd', 'centers': centers, 'sigmas': sigmas_sel, 'coeffs': coeffs})

    print(f"  wnd   {e+1}/{params_wnd['n_experts']} | rang={mask.sum():3d}/{k} | "
          f"loss={loss_data+loss_reg:.6f} (data={loss_data:.6f} reg={loss_reg:.6f}) | "
          f"{cls_str(f_te, y_test)} | {times_p1[-1]:.1f}s")

F_train = np.column_stack(F_train_list)
F_test = np.column_stack(F_test_list)
n_exp = len(experts)
expert_names = [f"{ex['type']}{i+1}" for i, ex in enumerate(experts)]
# ==========================================================
# Vote majoritaire
# ==========================================================

vote_train = np.sign(F_train)
vote_train[vote_train == 0] = 1

vote_test = np.sign(F_test)
vote_test[vote_test == 0] = 1

pred_vote_train = vote_train.sum(axis=1)
pred_vote_test = vote_test.sum(axis=1)

print("\nVote majoritaire")
print(f"  train : {cls_str(pred_vote_train, y_train)}")
print(f"  test  : {cls_str(pred_vote_test, y_test)}")

w = 1.0 / (expert_losses + 1e-12)
w /= w.sum()

pred_weighted_train = F_train @ w
pred_weighted_test  = F_test @ w

print("\nVote pondéré (1/loss)")
print(f"  train : {cls_str(pred_weighted_train, y_train)}")
print(f"  test  : {cls_str(pred_weighted_test, y_test)}")


pred_vote_soft_train = F_train.mean(axis=1)
pred_vote_soft_test  = F_test.mean(axis=1)

print("\nVote moyen (scores)")
print(f"  train : {cls_str(pred_vote_soft_train, y_train)}")
print(f"  test  : {cls_str(pred_vote_soft_test, y_test)}")
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2 : Gram H entre TOUS les experts (gaussiens ET Wendland mélangés),
# directions PARTAGÉES — hyperparamètres (weights, lambda_reg, thresh_factor,
# n_G_H) INDÉPENDANTS de ceux utilisés en Phase 1 pour chaque type.
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nPhase 2 : Gram H entre {n_exp} experts ({params_gauss['n_experts']} gauss + "
      f"{params_wnd['n_experts']} wnd), n_G_H={params_phase2['n_G_H']}...")

rng_H = np.random.default_rng(seed=999)
idx_H = rng_H.choice(N_all, size=min(N_all, params_phase2["n_G_H"]), replace=False)
X_H = X_all[idx_H]; n_H = len(idx_H)

w0 = params_phase2["weights"].get(0, 0.); w1 = params_phase2["weights"].get(1, 0.)
w2 = params_phase2["weights"].get(2, 0.); w3 = params_phase2["weights"].get(3, 0.)
need2 = w2 != 0; need3 = w3 != 0

PHI_H = np.zeros((n_H, n_exp))
TR_H = np.zeros((n_H, n_exp)) if need2 else None
V_H = np.zeros((n_H, n_exp, d)) if need3 else None

for e, ex in enumerate(experts):
    diff = X_H[:, None, :] - ex['centers'][None, :, :]
    inv2 = 1.0/ex['sigmas']**2
    if ex['type'] == 'gauss':
        sq = np.sum(diff**2*inv2[None, :, :], axis=2)
        phi_k = np.exp(-sq/2)
        PHI_H[:, e] = phi_k@ex['coeffs']
        if need2:
            TR_k = (-np.sum(inv2, axis=1)[None, :] + np.sum(diff**2*inv2[None, :, :]**2, axis=2))*phi_k
            TR_H[:, e] = TR_k@ex['coeffs']
        if need3:
            Csum = -np.sum(inv2, axis=1)
            Dq = np.sum(diff**2*inv2[None, :, :]**2, axis=2)
            coefV = 2*inv2[None, :, :]**2 - (Csum[None, :, None]+Dq[:, :, None])*inv2[None, :, :]
            V_k = phi_k[:, :, None]*diff*coefV
            V_H[:, e, :] = np.einsum('nmd,m->nd', V_k, ex['coeffs'])
    else:  # wnd
        r = np.sqrt(np.sum(diff**2*inv2[None, :, :], axis=2))
        rp = np.maximum(1.-r, 0.)
        r_safe = np.maximum(r, 1e-9)
        p0k, p1k, p2k = WND_PHI(r, rp), WND_P1(r, rp), WND_P2(r, rp)
        PHI_H[:, e] = p0k@ex['coeffs']
        if need2 or need3:
            Q4 = np.sum(diff**2*inv2[None, :, :]**2, axis=2)
            S0 = np.sum(inv2, axis=1)
        if need2:
            TR_k = p2k*Q4/r_safe**2 + p1k*(S0[None, :]/r_safe - Q4/r_safe**3)
            TR_k = np.where(r > 1e-9, TR_k, 0.)
            TR_H[:, e] = TR_k@ex['coeffs']
        if need3:
            p3k = WND_P3(r, rp)
            dr = diff*inv2[None, :, :]/r_safe[:, :, None]
            dQ4 = 2*diff*inv2[None, :, :]**2
            V_k = (p3k[:, :, None]*dr*Q4[:, :, None]/r_safe[:, :, None]**2
                  + p2k[:, :, None]*(dQ4/r_safe[:, :, None]**2 - 2*Q4[:, :, None]*dr/r_safe[:, :, None]**3)
                  + p2k[:, :, None]*dr*(S0[None, :, None]/r_safe[:, :, None] - Q4[:, :, None]/r_safe[:, :, None]**3)
                  + p1k[:, :, None]*(-S0[None, :, None]*dr/r_safe[:, :, None]**2
                                     - dQ4/r_safe[:, :, None]**3
                                     + 3*Q4[:, :, None]*dr/r_safe[:, :, None]**4))
            V_k = np.where(r[:, :, None] > 1e-9, V_k, 0.)
            V_H[:, e, :] = np.einsum('nmd,m->nd', V_k, ex['coeffs'])

G_H = np.zeros((n_exp, n_exp))
if w0:
    G_H += w0*(PHI_H.T@PHI_H)/n_H
if need2:
    TRG_H = (TR_H.T@TR_H)/n_H
if need3:
    VVG_H = np.einsum('nid,njd->ij', V_H, V_H)/n_H

if w1 or need2 or need3:
    rng_dir = np.random.default_rng(seed=8000)
    MC1_sum = np.zeros((n_exp, n_exp)); MC2_sum = np.zeros((n_exp, n_exp)); MC3_sum = np.zeros((n_exp, n_exp))
    done = 0
    while done < params_shared["n_dirs"]:
        K = min(params_shared["batch_dirs"], params_shared["n_dirs"] - done)
        U = rng_dir.normal(size=(n_H, K, d))
        U /= np.linalg.norm(U, axis=2, keepdims=True)

        D1_agg = np.zeros((n_H, K, n_exp)) if w1 else None
        D2_agg = np.zeros((n_H, K, n_exp)) if (need2 or need3) else None
        D3_agg = np.zeros((n_H, K, n_exp)) if need3 else None

        for e, ex in enumerate(experts):
            diff = X_H[:, None, :] - ex['centers'][None, :, :]
            inv2 = 1.0/ex['sigmas']**2
            if ex['type'] == 'gauss':
                sq = np.sum(diff**2*inv2[None, :, :], axis=2)
                phi_k = np.exp(-sq/2)
                A = np.einsum('nmd,nkd,md->nmk', diff, U, inv2)
                B = np.einsum('nkd,md->nmk', U**2, inv2)
                gp = -A; gpp = -B
                if w1:
                    D1_k = gp*phi_k[:, :, None]
                    D1_agg[:, :, e] = np.einsum('nmk,m->nk', D1_k, ex['coeffs'])
                if need2 or need3:
                    D2_k = (gpp+gp**2)*phi_k[:, :, None]
                    D2_agg[:, :, e] = np.einsum('nmk,m->nk', D2_k, ex['coeffs'])
                if need3:
                    D3_k = (3*gp*gpp+gp**3)*phi_k[:, :, None]
                    D3_agg[:, :, e] = np.einsum('nmk,m->nk', D3_k, ex['coeffs'])
            else:  # wnd
                r = np.sqrt(np.sum(diff**2*inv2[None, :, :], axis=2))
                rp = np.maximum(1.-r, 0.)
                r_safe = np.maximum(r, 1e-9)
                p1k, p2k, p3k = WND_P1(r, rp), WND_P2(r, rp), WND_P3(r, rp)
                A = np.einsum('nmd,nkd,md->nmk', diff, U, inv2)
                B = np.einsum('nkd,md->nmk', U**2, inv2)
                r0 = r_safe[:, :, None]
                rprime = A/r0; rpprime = -A**2/r0**3 + B/r0
                if w1:
                    D1_k = p1k[:, :, None]*rprime
                    D1_agg[:, :, e] = np.einsum('nmk,m->nk', D1_k, ex['coeffs'])
                if need2 or need3:
                    D2_k = p2k[:, :, None]*rprime**2 + p1k[:, :, None]*rpprime
                    D2_agg[:, :, e] = np.einsum('nmk,m->nk', D2_k, ex['coeffs'])
                if need3:
                    rppprime = 3*A*(A**2 - B*r0**2)/r0**5
                    D3_k = p3k[:, :, None]*rprime**3 + 3*p2k[:, :, None]*rprime*rpprime + p1k[:, :, None]*rppprime
                    D3_agg[:, :, e] = np.einsum('nmk,m->nk', D3_k, ex['coeffs'])

        if w1:
            D1f = D1_agg.reshape(-1, n_exp); MC1_sum += D1f.T@D1f
        if need2:
            D2f = D2_agg.reshape(-1, n_exp); MC2_sum += D2f.T@D2f
        if need3:
            D3f = D3_agg.reshape(-1, n_exp); MC3_sum += D3f.T@D3f
        done += K

    if w1: G_H += w1*d*MC1_sum/(n_H*params_shared["n_dirs"])
    if need2:
        MC2 = MC2_sum/(n_H*params_shared["n_dirs"])
        G_H += w2*(d*(d+2)*MC2 - TRG_H)/2
    if need3:
        MC3 = MC3_sum/(n_H*params_shared["n_dirs"])
        G_H += w3*(d*(d+2)*(d+4)*MC3 - 9*VVG_H)/6

print(f"  G_H calculée ({n_H} pts, {params_shared['n_dirs']} directions partagées)")

n_loc = len(y_train)
m_vec = F_train.T@y_train/n_loc
q_vec = np.sum(F_train**2, axis=0)/n_loc
denom = q_vec + params_phase2["lambda_reg"]*np.diag(G_H)
bad_denom = denom <= 1e-14
if bad_denom.any():
    print(f"  [!] {bad_denom.sum()} expert(s) dégénéré(s) écarté(s)")
denom_safe = np.where(bad_denom, 1.0, denom)
expert_losses = np.mean(y_train**2) - m_vec**2/denom_safe
expert_losses = np.where(bad_denom, np.inf, expert_losses)

best_loss = expert_losses.min()
sel = expert_losses <= params_shared["k_loss"]*best_loss
sel_idx = np.where(sel)[0]
print(f"\n  Sélection experts (k_loss={params_shared['k_loss']}) : {sel.sum()}/{n_exp} retenus")
for i in range(n_exp):
    tag = "GARDÉ " if sel[i] else "écarté"
    print(f"    {expert_names[i]:6s} : loss={expert_losses[i]:.3e}  [{tag}]")

F_train_sel = F_train[:, sel]; F_test_sel = F_test[:, sel]
G_H_sel = G_H[np.ix_(sel_idx, sel_idx)]

n_tr = F_train_sel.shape[0]
M_H = (F_train_sel.T@F_train_sel)/n_tr + params_phase2["lambda_reg"]*G_H_sel
rhs_H = (F_train_sel.T@y_train)/n_tr
eigvals_H, eigvecs_H = np.linalg.eigh(M_H)
thresh_H = max(params_phase2["lambda_reg"]*params_phase2["thresh_factor"], 1e-10)
mask_H = eigvals_H > thresh_H
V_Hm = eigvecs_H[:, mask_H]; S_H = eigvals_H[mask_H]
alpha_sel = V_Hm@((V_Hm.T@rhs_H)/S_H)

alpha = np.zeros(n_exp); alpha[sel_idx] = alpha_sel

pred_H_train = F_train_sel@alpha_sel
pred_H_test = F_test_sel@alpha_sel

print(f"\n  rang H = {mask_H.sum()}/{sel.sum()} (sur {n_exp} experts)")
print(f"  train : {cls_str(pred_H_train, y_train)}   test : {cls_str(pred_H_test, y_test)}")

print(f"\n  Poids attribués à chaque expert (alpha) :")
for i in range(n_exp):
    tag = "" if sel[i] else "  (écarté, alpha=0)"
    print(f"    {expert_names[i]:6s} : alpha={alpha[i]:+.4f}{tag}")


# ══════════════════════════════════════════════════════════════════════════════
# COMPARAISONS : Ridge polynomial, SVM RBF, Ridge RBF
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.svm import SVC
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import GridSearchCV

print("\nCalibration Ridge polynomial...")
poly = PolynomialFeatures(degree=min(params_shared["deg_P"], 8), include_bias=False)
ridge_poly = Ridge(alpha=1e-8)
ridge_poly.fit(poly.fit_transform(X_train), y_train)
pred_ridgepoly_te = ridge_poly.predict(poly.transform(X_test))

print("Calibration SVM RBF (GridSearchCV)...")
param_grid_svm = {"C": [0.1, 1, 10, 100], "gamma": ["scale", 0.01, 0.1, 1]}
svm = GridSearchCV(SVC(kernel="rbf"), param_grid_svm, cv=5, n_jobs=-1)
svm.fit(X_train, y_train)
pred_svm_te = svm.decision_function(X_test)

print("Calibration Ridge RBF (GridSearchCV)...")
param_grid_kr = {"alpha": [1e-3, 1e-2, 1e-1, 1.0], "gamma": [0.001, 0.01, 0.1, 1]}
kr = GridSearchCV(KernelRidge(kernel="rbf"), param_grid_kr, cv=5, n_jobs=-1)
kr.fit(X_train, y_train)
pred_kr_te = kr.predict(X_test)

print("\n" + "="*80)
print("RÉSUMÉ")
print("="*80)
print(f"Sobolev H (mixte gauss+wnd, {sel.sum()}/{n_exp} experts) : {cls_str(pred_H_test, y_test)}")
print(f"Ridge polynomial                                  : {cls_str(pred_ridgepoly_te, y_test)}")
print(f"SVM RBF     (best={svm.best_params_})               : {cls_str(pred_svm_te, y_test)}")
print(f"Ridge RBF   (best={kr.best_params_})               : {cls_str(pred_kr_te, y_test)}")
print(f"\nn_train={params_shared['n_train']}  n_unlabeled={params_shared['n_unlabeled']}  "
      f"n_test={params_shared['n_test']}  n_dirs={params_shared['n_dirs']}")
print(f"temps phase 1 : {sum(times_p1):.1f}s total ({np.mean(times_p1):.1f}s/expert)")
print("\nAccuracy/AUC individuelles des experts :")
for i in range(n_exp):
    tag = "" if sel[i] else "  (écarté)"
    print(f"  {expert_names[i]:6s} : {cls_str(F_test[:,i], y_test)}  loss={expert_losses[i]:.3e}{tag}")

Génération du cloud sur {Q=0}...
  collectés:797/200 (cumulé 99.6%)
  collectés:19970/5000 (cumulé 99.9%)
  collectés:15183/3800 (cumulé 99.9%)
  X_train:(200, 5)  X_unlabeled:(3800, 5)
  cible binarisée au seuil médian=1.1697 : train (+1)=113/200  test (+1)=2530/5000
Profil Wendland C4 (ordre max requis, Phase 1 wnd ∪ Phase 2 = 3)

Phase 1 (gaussien) : 3 experts indépendants...
  gauss 1/3 | rang=454/800 | loss=0.451013 (data=0.360289 reg=0.090724) | acc=0.8248 AUC=0.8947 | 40.8s
  gauss 2/3 | rang=434/800 | loss=0.447599 (data=0.357750 reg=0.089849) | acc=0.8230 AUC=0.8956 | 40.5s
  gauss 3/3 | rang=452/800 | loss=0.443574 (data=0.352358 reg=0.091216) | acc=0.8254 AUC=0.8978 | 40.6s

Phase 1 (Wendland) : 3 experts indépendants...
  wnd   1/3 | rang=779/1000 | loss=0.820667 (data=0.722354 reg=0.098312) | acc=0.7910 AUC=0.8577 | 71.4s
